# exp02 — Train baseline Transformer trên Multi-VSL 200

**Mở trên Colab:** https://colab.research.google.com/github/minKasent/signbridge/blob/main/training/experiments/exp02_train_baseline.ipynb

**Chạy SAU KHI exp01 đã trích landmark xong** (Drive có `datn/landmarks/{train,val,test}`).

- **Runtime: bật GPU T4** (Thời gian chạy → Thay đổi loại thời gian chạy → T4 GPU).
- Thời gian: ~30-60 phút cho 40 epoch trên T4.
- Model + toàn bộ pipeline này đã được smoke-test trên dữ liệu giả (loss giảm, NO_SIGN 96%)
  nên lỗi nếu có sẽ nằm ở dữ liệu, không phải code.
- Kết quả: `sign_model.onnx` + `labels.json` lưu vào Drive — copy vào `apps/ml/model/`
  trên máy là trang /translate nhận diện ký hiệu THẬT.

In [ ]:
# Cell 1 — Chuẩn bị (~1 phút)
%pip install -q wandb onnxscript
!git clone -q https://github.com/minKasent/signbridge.git /content/signbridge 2>/dev/null || (cd /content/signbridge && git pull -q)

import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CHƯA BẬT — bật T4 rồi chạy lại")

from google.colab import drive
drive.mount("/content/drive")

import os
DATA = "/content/drive/MyDrive/datn/landmarks"
for split in ["train", "val", "test"]:
    n = sum(len(files) for _, _, files in os.walk(f"{DATA}/{split}"))
    print(f"{split}: ~{n} file")

In [ ]:
# Cell 2 — Đăng nhập W&B (dán key từ file .env trên máy khi được hỏi)
import wandb
wandb.login()

In [ ]:
# Cell 3 — Train (~30-60 phút trên T4; theo dõi biểu đồ trực tiếp trên wandb.ai)
OUT = "/content/drive/MyDrive/datn/models/exp02"
!cd /content/signbridge/training/model && python train.py \
    --data "/content/drive/MyDrive/datn/landmarks" \
    --out "{OUT}" \
    --epochs 40 --batch 64 --workers 2 \
    --wandb --run-name exp02-baseline-multivsl200

In [ ]:
# Cell 4 — Kiểm tra ONNX + tải về máy
import json, os
OUT = "/content/drive/MyDrive/datn/models/exp02"
meta = json.load(open(f"{OUT}/labels.json", encoding="utf-8"))
print(f"Lớp: {len(meta['labels'])} | val top1: {meta['val_top1']:.3f} | test top1: {meta['test_top1']:.3f}")
print(f"ONNX: {os.path.getsize(f'{OUT}/sign_model.onnx')/1e6:.1f} MB")

from google.colab import files
files.download(f"{OUT}/sign_model.onnx")
files.download(f"{OUT}/labels.json")
print("\nĐặt 2 file này vào d:/Khoa/DATN/apps/ml/model/ rồi khởi động lại ML service —")
print("trang /translate sẽ nhận diện ký hiệu thật.")

## Sau baseline — các thí nghiệm cho chương 4 của báo cáo

Chạy lại Cell 3 với tham số khác, mỗi lần một `--run-name` riêng (W&B tự vẽ bảng so sánh):
- `--d-model 128 --layers 2` (model nhỏ — nhanh hơn, kém hơn bao nhiêu?)
- `--d-model 256 --layers 6` (model to — có đáng không?)
- `--no-sign-per-class 0` (bỏ lớp NO_SIGN — chứng minh vì sao cần nó)
- `--window 16` vs `--window 48` (cửa sổ ngắn/dài)

Nhãn hiện là **số 0-198** (repo tác giả không kèm tên ký hiệu) — nếu Cell 2 của exp01
tìm thấy file từ vựng trong folder Drive thì báo để cập nhật từ điển + LLM prompt.